# D-RECIPE: Dynamic Temporal Knowledge Graph Forecasting

**Continual LLM-based TKG completion with streaming edge ingestion.**

> Before running: go to **Runtime → Change runtime type** and select **GPU (T4 or A100)**.

---

### How datasets work here

GitHub rejects files over 100 MB, so the datasets (ICEWS14, ICEWS18, GDELT, YAGO) live on
**your Google Drive** instead. You only upload them once — Colab mounts Drive every session
and reads directly from it via a symlink (no copying, no re-uploading).

**One-time setup on your computer:**
1. Open [drive.google.com](https://drive.google.com)
2. Create the folder `My Drive/D-RECIPE/data/original/`
3. Upload your dataset folders into it:
   ```
   My Drive/D-RECIPE/data/original/
   ├── icews14/   ← train.txt, valid.txt, test.txt, entity2id.json, relation2id.json, ts2id.json
   ├── icews18/
   ├── GDELT/
   └── YAGO/
   ```
4. Done — never do this again.

---
**Steps in this notebook:**
1. Mount Google Drive
2. Clone repo & install dependencies
3. Link datasets from Drive into the repo
4. Authenticate HuggingFace (for LLaMA)
5. Preprocess data
6. Configure & train
7. Inference & evaluation
8. Generate plots
9. Ablation study (optional)

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# ── Paths on your Drive ───────────────────────────────────────────────────────
DRIVE_ROOT    = '/content/drive/MyDrive/D-RECIPE'
DRIVE_DATA    = f'{DRIVE_ROOT}/data/original'   # where YOUR datasets live
DRIVE_RESULTS = f'{DRIVE_ROOT}/results'         # where outputs will be saved

os.makedirs(DRIVE_DATA,    exist_ok=True)
os.makedirs(DRIVE_RESULTS, exist_ok=True)

# Verify datasets are present
found = [d for d in ['icews14','icews18','GDELT','YAGO'] if os.path.isdir(f'{DRIVE_DATA}/{d}')]
missing = [d for d in ['icews14','icews18','GDELT','YAGO'] if d not in found]

print('Drive mounted at /content/drive')
print(f'Datasets found  : {found if found else "NONE"}')
if missing:
    print(f'Datasets MISSING: {missing}')
    print(f'  → Upload them to: {DRIVE_DATA}/')
else:
    print('All 4 datasets present. Ready to go!')

## 2. Clone Repository & Install Dependencies

In [ ]:
REPO_URL = 'https://github.com/saniemacdube93/forecast.git'  # update if your URL differs
REPO_DIR = '/content/forecast'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print('Repo already cloned — pulling latest')
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}
!ls

In [ ]:
# Install dependencies (~3-5 min on first run, cached on reconnect)
!pip install -q -r requirements_dynamic.txt
print('Done.')

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU :', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

## 3. Link Datasets from Drive → Repo

A symlink makes the Drive folder appear as `data/` inside the cloned repo.
**No copying, no re-uploading** — reads straight from Drive every session.

In [ ]:
REPO_DATA = f'{REPO_DIR}/data'

# Remove any stale symlink or empty folder from a previous session
if os.path.islink(REPO_DATA):
    os.unlink(REPO_DATA)
elif os.path.isdir(REPO_DATA) and not os.listdir(REPO_DATA):
    os.rmdir(REPO_DATA)

if not os.path.exists(REPO_DATA):
    os.symlink(f'{DRIVE_ROOT}/data', REPO_DATA)
    print(f'Symlink created: {REPO_DATA} → {DRIVE_ROOT}/data')
else:
    print(f'data/ already linked or populated: {REPO_DATA}')

# Confirm
!ls {REPO_DATA}/original/

## 4. HuggingFace Authentication

LLaMA models require you to accept their licence on HuggingFace first:
- LLaMA-2: https://huggingface.co/meta-llama/Llama-2-7b-hf
- LLaMA-3: https://huggingface.co/meta-llama/Meta-Llama-3-8B

**Recommended:** add your token as a Colab Secret named `HF_TOKEN`
(click the 🔑 key icon in the left sidebar).

In [ ]:
from huggingface_hub import login
from google.colab import userdata

try:
    login(token=userdata.get('HF_TOKEN'), add_to_git_credential=False)
    print('Logged in via Colab Secret HF_TOKEN.')
except Exception:
    hf_token = ''  # ← paste your token here as fallback
    if hf_token:
        login(token=hf_token, add_to_git_credential=False)
        print('Logged in via pasted token.')
    else:
        print('WARNING: No HuggingFace token. LLaMA download will fail.')

## 5. Preprocess Data

Run once per dataset. Outputs are saved back to Drive so you don't repeat this.

In [ ]:
# ← Set which datasets you want to preprocess (can be a subset)
DATASETS_TO_PREP = ['icews14', 'icews18', 'GDELT', 'YAGO']

%cd {REPO_DIR}/data_utils
for ds in DATASETS_TO_PREP:
    print(f'\n===== Preprocessing {ds} =====')
    !python retrieve.py --dataset {ds} --retrieve_type weighted
%cd {REPO_DIR}

## 6. Configure & Train

Edit the config cell, then run the debug cell first to confirm everything works before the full run.

In [ ]:
# ── Choose your dataset & model ───────────────────────────────────────────────
DATASET    = 'icews14'                    # icews14 | icews18 | GDELT | YAGO
MODEL_NAME = 'meta-llama/Llama-2-7b-hf'  # or meta-llama/Meta-Llama-3-8B

DATA_PATH   = './data/original/'
RESULTS_DIR = f'{DRIVE_RESULTS}/{DATASET}_llama2'
os.makedirs(RESULTS_DIR, exist_ok=True)

# ── Stream simulation ─────────────────────────────────────────────────────────
STREAM_CHUNK_SIZE = 500
SEED_CHUNKS       = 5
N_CHUNKS          = 20
EPOCHS            = 5

# ── Continual learning ────────────────────────────────────────────────────────
EWC_LAMBDA   = 0.1
KD_GAMMA     = 0.3
KD_TEMP      = 2.0
REPLAY_RATIO = 0.3
BUFFER_SIZE  = 5000

# ── Hardware ──────────────────────────────────────────────────────────────────
DEVICE             = 'cuda'
MICRO_BATCH_SIZE   = 2
GRAD_CHECKPOINTING = 1

print(f'Dataset    : {DATASET}')
print(f'Model      : {MODEL_NAME}')
print(f'Results dir: {RESULTS_DIR}')

In [ ]:
# ── Debug run (~5 min) — run this first to confirm the pipeline works ─────────
!python dynamic_main.py \
    --DATASET            {DATASET} \
    --MODEL_NAME         {MODEL_NAME} \
    --DATA_PATH          {DATA_PATH} \
    --RESULTS_DIR        {RESULTS_DIR}/debug \
    --STREAM_CHUNK_SIZE  100 \
    --SEED_CHUNKS        1 \
    --N_CHUNKS           3 \
    --EPOCHS             1 \
    --DEVICE             {DEVICE} \
    --DEBUG

In [ ]:
# ── Full training run ─────────────────────────────────────────────────────────
!python dynamic_main.py \
    --DATASET                {DATASET} \
    --MODEL_NAME             {MODEL_NAME} \
    --DATA_PATH              {DATA_PATH} \
    --RESULTS_DIR            {RESULTS_DIR} \
    --STREAM_CHUNK_SIZE      {STREAM_CHUNK_SIZE} \
    --SEED_CHUNKS            {SEED_CHUNKS} \
    --N_CHUNKS               {N_CHUNKS} \
    --EPOCHS                 {EPOCHS} \
    --EWC_LAMBDA             {EWC_LAMBDA} \
    --KD_GAMMA               {KD_GAMMA} \
    --KD_TEMPERATURE         {KD_TEMP} \
    --REPLAY_RATIO           {REPLAY_RATIO} \
    --BUFFER_SIZE            {BUFFER_SIZE} \
    --DEVICE                 {DEVICE} \
    --MICRO_BATCH_SIZE       {MICRO_BATCH_SIZE} \
    --GRADIENT_CHECKPOINTING {GRAD_CHECKPOINTING}

## 7. Inference & Evaluation

In [ ]:
CHECKPOINT  = f'{RESULTS_DIR}/checkpoints/final_model.pt'
OUTPUT_FILE = f'{RESULTS_DIR}/predictions.json'

!python dynamic_inference.py \
    --DATASET     {DATASET} \
    --CHECKPOINT  {CHECKPOINT} \
    --MODEL_NAME  {MODEL_NAME} \
    --DATA_PATH   {DATA_PATH} \
    --OUTPUT_FILE {OUTPUT_FILE} \
    --DEVICE      {DEVICE} \
    --EVAL

## 8. Generate & Display Plots

In [ ]:
PLOTS_DIR    = f'{RESULTS_DIR}/plots'
METRICS_FILE = f'{RESULTS_DIR}/metrics.json'
os.makedirs(PLOTS_DIR, exist_ok=True)

!python visualize_results.py \
    --results_file {METRICS_FILE} \
    --output_dir   {PLOTS_DIR}

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import glob

plot_files = sorted(glob.glob(f'{PLOTS_DIR}/*.png'))
if not plot_files:  # fall back to pre-generated repo plots
    plot_files = sorted(glob.glob(f'{REPO_DIR}/results/plots/*.png'))

for path in plot_files:
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.imshow(mpimg.imread(path))
    ax.axis('off')
    ax.set_title(os.path.basename(path), fontsize=11)
    plt.tight_layout()
    plt.show()

## 9. Ablation Study (Optional)

Remove one component at a time to measure its contribution.

In [ ]:
# Options: no_ewc | no_replay | no_kd | no_filtering | naive_finetune
ABLATION = 'no_ewc'

!python dynamic_main.py \
    --DATASET            {DATASET} \
    --MODEL_NAME         {MODEL_NAME} \
    --DATA_PATH          {DATA_PATH} \
    --RESULTS_DIR        {DRIVE_RESULTS}/ablation_{ABLATION} \
    --STREAM_CHUNK_SIZE  {STREAM_CHUNK_SIZE} \
    --SEED_CHUNKS        {SEED_CHUNKS} \
    --N_CHUNKS           {N_CHUNKS} \
    --EPOCHS             {EPOCHS} \
    --ABLATION           {ABLATION} \
    --DEVICE             {DEVICE}

## 10. Preview Plots Without Training

Generates all 11 figure types from the paper's baseline numbers — **no GPU or LLaMA download needed**.
Good for checking your Colab environment is working before a long run.

In [ ]:
MOCK_DIR = f'{DRIVE_RESULTS}/mock_plots'
os.makedirs(MOCK_DIR, exist_ok=True)

!python visualize_results.py --mock --output_dir {MOCK_DIR}

for path in sorted(glob.glob(f'{MOCK_DIR}/*.png')):
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.imshow(mpimg.imread(path))
    ax.axis('off')
    ax.set_title(os.path.basename(path), fontsize=11)
    plt.tight_layout()
    plt.show()